### Chapter 4 - Transforming Data
Reviewing Extract-Transform-Load (ETL) processes
#### Importing and assembling data
**Bad practices:**

- Manually deleting rows at the beginning of the file in spreadsheet software which fails to preserve the edit trail
- The number of columns differs between two files that are going to be appended. Deleting or adding columns fails to preserve the edit trail
- Changing column names when they are not desired
- Overwriting the original file which can have errors with no hope of preserving the original data
- Manual edits are learned and shared orally, not as codified procedure

We are going to load and assemble data from datasets which have some quirks. Goal is to assemble a gas price dataset for two US ports from the Energy Information Administration (EIA). We will load a CSV of gas prices for the US Golf Coast and two worksheets from an Excel workbook for NY Harbor gas prices. The columnms need to be renamed for consistency, then the datasets need to be joined together into a daily time series with three columns: `date`, `gulf_price`, and `ny_price`. Finally, we then save the processed data as a CSV and calculate a correlation coefficient between two gas price variables.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import os
os.getcwd()

'c:\\Users\\debro\\OneDrive\\Documents\\Public Policy\\Python Exercises'

In [8]:
# Load a CSV with the Gulf Gas Prices
gulf = pd.read_csv("../Data-Science-for-Public-Policy-data-sets/data/doe_usgulf.csv")
# Rename the columns
gulf.columns = ["date", "gulf_price"]
# View the first few rows of the Data
gulf.head(10)

,date,gulf_price
0,1/2/14,2.515
1,1/3/14,2.493
2,1/6/14,2.515
3,1/7/14,2.556
4,1/8/14,2.545
5,1/9/14,2.528
6,1/10/14,2.605
7,1/13/14,2.544
8,1/14/14,2.528
9,1/15/14,2.551


In [17]:
# Load the Excel workbook with NY data
ny1 = pd.read_excel("../Data-Science-for-Public-Policy-data-sets/data/doe_ny.xlsx", sheet_name="part1", skiprows=1).iloc[:, :2]
ny2 = pd.read_excel("../Data-Science-for-Public-Policy-data-sets/data/doe_ny.xlsx", sheet_name="part2", skiprows=1).iloc[:, :2]
# Change the column names
ny1.columns = ny2.columns = ["date", "ny_price"]
# Combine the two DataFrames
ny = pd.concat([ny1, ny2], axis=0, ignore_index=True)
# View the first few rows of the Data
ny.head(10)

,date,ny_price
0,2014-01-02,2.718
1,2014-01-03,2.671
2,2014-01-06,2.678
3,2014-01-07,2.704
4,2014-01-08,2.684
5,2014-01-09,2.667
6,2014-01-10,2.693
7,2014-01-13,2.643
8,2014-01-14,2.636
9,2014-01-15,2.639


In [18]:
gulf.describe()

,gulf_price
count,959.000000
mean,1.744842
std,0.532234
min,0.812000
25%,1.379000
50%,1.562000
75%,1.941000
max,2.953000


In [19]:
ny.describe()

,date,ny_price
count,959,959.000000
mean,2015-11-27 15:03:56.496350464,1.814097
min,2014-01-02 00:00:00,0.947000
25%,2014-12-13 12:00:00,1.449500
50%,2015-11-25 00:00:00,1.604000
75%,2016-11-07 12:00:00,2.042000
max,2017-10-23 00:00:00,3.023000
std,NaN,0.544024


We can bind the two datasets together since they have the same amount of rows. However, we probably want to merge on the columns on `date`. If you were to inspect the datasets, they would have the same data, just in a different datetime format. We can construct a new DataFrame using `pd.concat` and use the `to_csv` function to output the dataset to a CSV.

In [21]:
gas_prices = pd.concat([gulf, ny['ny_price']], axis=1)
gas_prices.to_csv("../data/gas_prices.csv", index=False)
gas_prices.to_json("../data/gas_prices.json", orient="records", lines=True)

#### Manipulating Values
Sometimes we need to clean up data and process the data before analyzing.

In [13]:
# import regular expressions module
import re
import numpy as np
# Create a list of "budget" strings
budget = [
    "Captain's Log, Stardate 1551.8. I have $10.20 for a big galactic map.",
    "The ensign has $1.20 in her pocket.",
    "The ExO has $0.25 left after paying for overpriced warp core fuel.",
    "Chief medical officer is the high roller with $53,13."
]
# Remove the commas for easier processing
bstr = [b.replace(",",".") for b in budget]
print(bstr)
# Extract the dollar amounts using regular expressions
amounts = [float(str(re.findall(r"\$(?:\d{1,3}(?:,\d{3})*|\d+)(?:\.\d{2})?", b)[0])[1:]) for b in bstr]
print(amounts)
# Calcaulate the total amount
print("Total amount available for a galactic big mac is: ${:.2f}".format(np.sum(amounts)))

["Captain's Log. Stardate 1551.8. I have $10.20 for a big galactic map.", 'The ensign has $1.20 in her pocket.', 'The ExO has $0.25 left after paying for overpriced warp core fuel.', 'Chief medical officer is the high roller with $53.13.']
[10.2, 1.2, 0.25, 53.13]
Total amount available for a galactic big mac is: $64.78


#### Text manipulation functions
*Find and replace* funcationality are common in word processing and spreadsheet software, but are not particularly efficient with complex string patterns. Here's a table with text manipulation functions:

| Description | `re`/Base Python | `pandas` |
| :---------- | :----- | :----- |
| Returns either the index position of a matched string or the</br>string containing the matched portion. | `re.search` | `.str.contains()` |
| Returns a logical vector indicating if a matched string was</br>found. | `[<pattern> in i for i <list>]`</br>`[bool(re.search(<pattern>, i)) for i in <list>]` | `.str.contains()` |
| Searches for a specified pattern and replaces with user-specified</br>substring. | `re.sub` | `.str.replace('str1', 'str2', regex=True)` |
| Remove matched pattern from string. | `re.sub` | `.str.replace('str1', 'str2', regex=True)` |
| Returns the first position of matched patterns in a string | `re.sub` | `.str.replace('str1', 'str2', regex=True)` |
| Returns the position of all matched patterns in a string | `re.search('pattern', String).start()` | `.str.find('pattern')` |
| Extract substring based on matched pattern. | `re.search(pattern, text).group()` | `.str.extract()` |
| Splits strings into a list of values based on a delimiter. | `re.split(pattern, text)` | `.str.split('pattern')` |
| Extract substring based on start and end positions | `string[start_idx:end_idx]` | `.str.slice(start, stop, step)` |
| Trim whitespace on either end of string (excessive spaces) | `re.sub(r'\s+', ' ', text)`</br>`text.strip()` | `.str.strip()` |
| Returns number of characters in string | `len(string)` | `.str.len()` |
| Returns the number of matched patterns | `len(re.findall(pattern, text))` | `.str.count(pattern)` |
| Convert to upper case | `result1 = re.sub(r'\b\w+\b', lambda match: match.group(0).upper(), text)` | `.str.upper()` |
| Convert to lower case | `result1 = re.sub(r'\b\w+\b', lambda match: match.group(0).lower(), text)` | `.str.lower()` |
| Convert to title case | `re.sub(r"[A-Za-z]+(\'[A-Za-z]+)?", lambda word: word.group(0).capitalize(), string)` | `.str.title()` |
| Pad string (e.g., add leading zeros to string) | `string.ljust(width, fillchar)`</br>`string.rjust(width, fillchar)`</br>`string.center(width, fillchar)` | `.str.pad(side, width, fillchar)` |

#### Regular Expressions (RegEx)
Regex expressions which dictate text patterns are the secret to manipulations.
1. Alternatives (e.g., OR searches) can be surfaced by using a pipe `|`
2. Extent of a search is denoted by parentheses `()`.
3. A search for one specific character should be placed between square brackes `[]`
4. The length of a match is specified using curly brackets `{}`

For example, in NYC, *Broadway* can be written and abbreviated in a number of ways.

In [19]:
# Put together some potential Broadway street name variations and non-Brodway street names
streets = [
    'Bruckner Blvd',
    'Bowery',
    'Broadway',
    'Bway',
    'Bdway',
    'Broad Street',
    'Bridge Street',
    "B'way"
]
# Search for two specific options using regular expressions
options = [s for s in streets if re.search(r'Broadway|Bdway', s)]
# Search for two specific variations using regular expressions
variations = [s for s in streets if re.search(r"B(road|')way", s)]
# Search for cases where either d or apostrophe is between B and way
apostrophe = [s for s in streets if re.search(r"B[d']way", s)]
# Get all Broadway variations even the apostrophe one
all_bway = [s for s in streets if re.search(r"B(road|d|')way", s)]
all_bway

['Broadway', 'Bdway', "B'way"]

**Escaped characters:**
- `\n` new line
- `\r` carriage return
- `\t` tab
- `\'` single quote when a string enclosed in single quotes
- `\"` double quote when a string is enclosed in double quotes
- `\\.` period. Otherwise, un-escaped periods indicates searches for any single character.
- `\\$` dollar sign. A dollar sign without backslashes indicates to find patterns at the end of a string.

**Character classes:** a *character class* or *character set* is used to identify specific characters within a string. One or more of the following character classes can do the job:
- R character class - `[:punct:]`: is a catch all for amny and all punctuation such as periods, commas, semicolons, etc. Enclose specific characters you want to remove by enclosing the characters inside of two brackets, e.g., `[<>,']`
- R character class - `[:alpha:]`: alphabetic characters such as a, b, c, etc. With other languages, searches for any letter combinations are denoted as `[A-Z]` for upper case characters, `[a-z]` for lower case, and `[A-z]` for mixed case.
- R character class - `[:digit:]`: Numerical values, and for other languages, it is written as `\\d` or `[0-9]`. For any non-digit, write `\\D`
- R character class - `[:alnum:]`: Alphanumeric characters. Indicated using to as `[0-9A-Za-z]` or `\\w`. For any non-alphanumeric character, use `\\W`
- R character class - `[:space:]`: Spaces such as tabs, carriage returns, etc. For any white space, use `\\s`. For any non-whitespace character, use `\\S`.
- R character class - `[:graph:]`: Human readable characters including alphanumeric and punctuation
- R character class - `\\b`: Used to denote whole words. `\\b` should be placed before and after a regex pattern. For example, `\\b\\w{10}\\bb` indicates a 10 letter word.

**Quantifiers**: indicates length of patterns to help narrow a search
- `{n}` match pattern *n* times for the preceding character class, e.g., `\\d{4}`
- `{n, m}` match pattern at least *m* times and not more than *n* times, e.g., `\\d{1,4}` looks for a number between 1 and 4 digits long
- `{n, }` match pattern at least *n* times, e.g. `\\d{4,}` looks for a number at least 4 digits long
- `*` Wildcard, or match at least 0 times
- `+` Match at least once
- `?` Match at most once

In [52]:
# Practice with pandas string methods
import pandas as pd
# Put together a list of dates inside strings
big_dates = pd.Series([
    "Octavian became Augustus on 16 Jan 27 BCE",
    "In the year 2000, a computer bug was expected to topple society.",
    "In 5400000000 years, our sun will become a red dwarf."
])
# Pull a nine digit number
big_dates.str.extract(r'(\d{9})')
# Extract a four digit substring that is flanked by empty value at either end
big_dates.str.extract(r'(^|\D)(\d{4})(\D|$)')
# Match a date that follows 16 January 27 BCE
big_dates.str.extract(r'(\d{1,2} \w+ \d{2} \w+)')

,0
0,16 Jan 27 BCE
1,NaN
2,NaN


**Positions.** Regex also builds functionality to search for patterns based on location of a substring, such as at the start or end of a string. There are a few position matching patterns, but these are the two main ones:
- `$`: search at the end of a string
- `^`: start of string when placed at the beginning of a regex pattern

In [55]:
# Using extract to get three headlines that contain 'May'
headlines = pd.Series([
    "May to deliver speech on Brexit",
    "Pound falls with May's comments",
    "May: Brexit plans to be laid out in new year"
])
# Extract any series which contain May at the beginning of the string
print(headlines.str.extract(r'^(May.*)'))
# Extract any series which contain Brexit at the end of the string
print(headlines.str.extract(r'(.*Brexit)$'))

                                              0
0               May to deliver speech on Brexit
1                                           NaN
2  May: Brexit plans to be laid out in new year
                                 0
0  May to deliver speech on Brexit
1                              NaN
2                              NaN


#### DIY: WOrking with PII
We are practicing some data privacy examples. Legislation by EU, the General Data Protection Regulation or GDPR, requires companies to protect personal data of EU citizens associated with transactions conducted in the EU. Work at the Census Bureau must apply disclousure avoidance practices in order so individuals cannot be identified.

**Redaction.** Clean the statements with PII

In [57]:
# Example strings with PII data
statement = pd.Series([
    "John Doe (SSN: 012-34-5678) has $2303 in savings in his account.",
    "Georgette Smith (SSN: 987-65-4321) owes $323 to the IRS.",
    "Alexander Doesmith (SSN: 098-76-5432) has a $10 library fine for overdue books."
])
# Remove SSNs, Names, and dollar amounts
pii_redact = statement.str.replace(r'\(SSN: \d{3}-\d{2}-\d{4}\)', '(SSN: XXX-XX-XXXX)', regex=True).str.replace(r'[A-Z][a-z]+ [A-Z][a-z]+', 'XXXXX', regex=True).str.replace(r'\$\d+(?:\.\d{2})?', '$XX', regex=True)
print(pii_redact)

0    XXXXX (SSN: XXX-XX-XXXX) has $XX in savings in...
1        XXXXX (SSN: XXX-XX-XXXX) owes $XX to the IRS.
2    XXXXX (SSN: XXX-XX-XXXX) has a $XX library fin...
dtype: object


Can we also extract this data in a table that looks like:

| Name | SSN | Money |
| :--- | :-- | :---- |
| John Doe | 012-34-5678 | $2303 |
| Georgette Smith | 987-65-4321 | $323 |
| Alexander Doesmith | 098-76-5432 | $10 |

In [58]:
pii_df = pd.DataFrame({
    "Name": statement.str.extract(r'([A-Z][a-z]+ [A-Z][a-z]+)')[0],
    "SSN": statement.str.extract(r'\(SSN: (\d{3}-\d{2}-\d{4})\)')[0],
    "Money": statement.str.extract(r'\$(\d+(?:\.\d{2})?)')[0]
})
pii_df

,Name,SSN,Money
0,John Doe,012-34-5678,2303
1,Georgette Smith,987-65-4321,323
2,Alexander Doesmith,098-76-5432,10


#### Working with dates
Most programming languages recognize dates, so we are going to get dates with 4 different formats and do some calculations.

In [70]:
# Import datetime module
import datetime as dt
# Examples of dates
d0 = dt.datetime.strptime("2010-08-20", "%Y-%m-%d")
d1 = dt.datetime.strptime("01/20/2010", "%m/%d/%Y")
d2 = dt.datetime.strptime("01/20/2010 00:00 AM", "%m/%d/%Y %H:%M %p")
d3 = dt.datetime.strptime("20100101", "%Y%m%d")
# See a calculation of the difference between two dates
print(d1-d3)
# Exrtact a year and quarter from a date
print("`d0` Year: {}, Quarter: {}".format(d1.year, (d1.month-1)//3 + 1))

19 days, 0:00:00
`d0` Year: 2010, Quarter: 1


The rest of the chapter deals with matrix data manipulations, iteration, conditionals, and defining functions. I am going to finish the chapter here.